# SQLite

[Video](https://youtu.be/Y5zdZpHPjvo)

Wir arbeiten mit der normalisierten Datenbank zur LK-Wahl aus dem Datenbank-Kapitel:

Lehrer(<u>Kuerzel</u>, Lehrer) <br>
Kurse(<u>KID</u>, Kuerzel↑, Stunden) <br>
Schueler(<u>SNr</u>, Name, Vorname) <br>
LKWahl(<u>SNr↑</u>, <u>KID↑</u>)



<table style="border-collapse: separate; border-spacing: 0; margin-left:0; margin-right:auto;">
<tr valign="top">

<td>
<table>
<tr>
    <th colspan="2">LKWahl</th>
</tr>
<tr>
    <th>SNr</th>
    <th>KID</th>
</tr>
<tr><td>120</td><td>M1</td></tr>
<tr><td>120</td><td>Ph1</td></tr>
<tr><td>121</td><td>Bio1</td></tr>
<tr><td>121</td><td>Ph1</td></tr>
<tr><td>122</td><td>M2</td></tr>
<tr><td>122</td><td>Ge2</td></tr>
</table>
</td>

<td style="padding-left:80px;">
<table>
<tr>
    <th colspan="3">Schueler</th>
</tr>
<tr>
    <th>SNr</th>
    <th>Name</th>
    <th>Vorname</th>
</tr>
<tr><td>120</td><td>Riedberg</td><td>Malte</td></tr>
<tr><td>121</td><td>Dreis</td><td>Lena</td></tr>
<tr><td>122</td><td>Rödding</td><td>Maike</td></tr>
</table>
</td>

<td style="padding-left:80px;">
<table>
<tr>
    <th colspan="3">Kurse</th>
</tr>
<tr>
    <th>KID</th>
    <th>Kuerzel</th>
    <th>Stunden</th>
</tr>
<tr><td>M1</td><td>SC</td><td>5</td></tr>
<tr><td>Ph1</td><td>SC</td><td>5</td></tr>
<tr><td>Bio1</td><td>DA</td><td>5</td></tr>
<tr><td>M2</td><td>AD</td><td>5</td></tr>
<tr><td>Ge2</td><td>GE</td><td>5</td></tr>
</table>
</td>

<td style="padding-left:80px;">
<table>
<tr>
    <th colspan="2">Lehrer</th>
</tr>
<tr>
    <th>Kuerzel</th>
    <th>Lehrer</th>
</tr>
<tr><td>SC</td><td>Schulte</td></tr>
<tr><td>DA</td><td>Dallmeyer</td></tr>
<tr><td>AD</td><td>Adam</td></tr>
<tr><td>GE</td><td>Gebauer</td></tr>
</table>
</td>

</tr>
</table>

- **SQLite** ist das **Datenbankmanagementsystem (DBMS)**. Es verwaltet die Daten, prüft die Regeln (z. B. Primärschlüssel) und führt unsere Anweisungen aus.
- Das Python-Modul **`sqlite3`** ist die **Datenbankschnittstelle**. Darüber schickt unser Python-Programm Anweisungen an das DBMS.
- **SQL** (*Structured Query Language*) ist die **Sprache**, in der wir die Anweisungen formulieren.
- Die **Datenbank** selbst steht bei SQLite in einer einzigen Datei, hier `lkwahl.db`.


```
Python-Programm  →  sqlite3 (Schnittstelle)  →  SQLite (DBMS)  →  lkwahl.db (Datei)
```

### Datenbank anlegen



##### Connection und Cursor

Zuerst bauen wir eine **Verbindung** (Connection) zur Datenbank auf. Über einen **Cursor** übermitteln wir dann die SQL-Anweisungen mit `execute()`. Jede Arbeit mit der Datenbank folgt diesem Muster:

In [ ]:
import sqlite3
con = sqlite3.connect("lkwahl.db")    # wird neu angelegt, falls sie noch nicht existiert
cur = con.cursor()

# hier kommen unsere SQL-Anweisungen hin

con.commit()        # falls Änderungen in der Datenbank gespeichert werden sollen
con.close()

`commit()` brauchen wir nur, wenn wir die Datenbank **verändern**. Ohne `commit()` gehen die Änderungen beim Schließen verloren.

##### CREATE TABLE

Mit `CREATE TABLE` legen wir eine Tabelle an. Für jede Spalte geben wir einen Namen und einen Datentyp an:

Die wichtigsten Datentypen beim Anlegen einer Tabelle sind für uns `INTEGER, REAL, TEXT`, 
das entspricht den Python-Datentypen `int, float, str`. 
Wir können zusätzliche Angaben machen, z.B. ob eine Spalte der Primärschlüssel ist, oder ein Fremdschlüssel oder ob die Spalte nicht leer bleiben darf.

**Beispiele:**

Schueler(<u>SNr</u>, Name, Vorname)  - das Schema entspricht der SQL-Anweisung:

```
CREATE TABLE Schueler (
    SNr INTEGER PRIMARY KEY,
    Name TEXT NOT NULL,
    Vorname TEXT NOT NULL
);
```

Lehrer(<u>Kuerzel</u>, Lehrer)  - das Schema entspricht der SQL-Anweisung:

```
CREATE TABLE Lehrer (
    Kuerzel TEXT PRIMARY KEY,
    Lehrer TEXT NOT NULL
);
```

Kurse(<u>KID</u>, Kuerzel↑, Stunden) - das Schema entspricht der SQL-Anweisung:

```
CREATE TABLE Kurse (
    KID TEXT PRIMARY KEY,                              
    Kuerzel TEXT NOT NULL,
    Stunden INTEGER,
    FOREIGN KEY (Kuerzel) REFERENCES Lehrer(Kuerzel)   
)
```

LKWahl(<u>SNr↑</u>, <u>KID↑</u>) - das Schema entspricht der SQL-Anweisung:

```
CREATE TABLE LKWahl (
    SNr INTEGER,
    KID TEXT,
    PRIMARY KEY (SNr, KID),
    FOREIGN KEY (SNr) REFERENCES Schueler(SNr),
    FOREIGN KEY (KID) REFERENCES Kurse(KID)
)
```

**Schreibweise:** SQL unterscheidet bei Schlüsselwörtern wie `CREATE TABLE`, `SELECT` oder `WHERE` nicht zwischen Groß- und Kleinschreibung, `select` funktioniert genauso wie `SELECT`. Üblich ist aber, Schlüsselwörter **groß** zu schreiben. 



##### Die Datenbank LKWahl aufbauen

Wir legen jetzt die vier Tabellen der normalisierten LK-Wahl an und füllen sie mit `INSERT INTO`. `execute()` kann immer nur **eine** SQL-Anweisung ausführen. Mit `executescript()` können wir mehrere Anweisungen auf einmal übergeben. Sie werden durch einen **Strichpunkt**
getrennt.

Mit `DROP TABLE IF EXISTS` löschen wir eine Tabelle, falls es sie schon gibt. So können wir die Zelle beliebig oft ausführen.

In [ ]:
import sqlite3

con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.executescript("""
DROP TABLE IF EXISTS LKWahl;
DROP TABLE IF EXISTS Schueler;
DROP TABLE IF EXISTS Kurse;
DROP TABLE IF EXISTS Lehrer;

CREATE TABLE Lehrer (
    Kuerzel TEXT PRIMARY KEY,
    Lehrer TEXT NOT NULL
);

CREATE TABLE Kurse (
    KID TEXT PRIMARY KEY,
    Kuerzel TEXT NOT NULL,
    Stunden INTEGER,
    FOREIGN KEY (Kuerzel) REFERENCES Lehrer(Kuerzel)
);

CREATE TABLE Schueler (
    SNr INTEGER PRIMARY KEY,
    Name TEXT NOT NULL,
    Vorname TEXT NOT NULL
);

CREATE TABLE LKWahl (
    SNr INTEGER,
    KID TEXT,
    PRIMARY KEY (SNr, KID),
    FOREIGN KEY (SNr) REFERENCES Schueler(SNr),
    FOREIGN KEY (KID) REFERENCES Kurse(KID)
);

INSERT INTO Lehrer VALUES ('SC', 'Schulte');
INSERT INTO Lehrer VALUES ('DA', 'Dallmeyer');
INSERT INTO Lehrer VALUES ('AD', 'Adam');
INSERT INTO Lehrer VALUES ('GE', 'Gebauer');

INSERT INTO Kurse VALUES ('M1', 'SC', 5);
INSERT INTO Kurse VALUES ('Ph1', 'SC', 5);
INSERT INTO Kurse VALUES ('Bio1', 'DA', 5);
INSERT INTO Kurse VALUES ('M2', 'AD', 5);
INSERT INTO Kurse VALUES ('Ge2', 'GE', 5);

INSERT INTO Schueler VALUES (120, 'Riedberg', 'Malte');
INSERT INTO Schueler VALUES (121, 'Dreis', 'Lena');
INSERT INTO Schueler VALUES (122, 'Rödding', 'Maike');

INSERT INTO LKWahl VALUES (120, 'M1');
INSERT INTO LKWahl VALUES (120, 'Ph1');
INSERT INTO LKWahl VALUES (121, 'Bio1');
INSERT INTO LKWahl VALUES (121, 'Ph1');
INSERT INTO LKWahl VALUES (122, 'M2');
INSERT INTO LKWahl VALUES (122, 'Ge2');
""")

con.commit()
con.close()
print('Datenbank erfolgreich erstellt.')

##### Den Inhalt einer Tabelle ansehen

Um zu prüfen, was in einer Tabelle steht, schicken wir eine **Abfrage** an das DBMS. `SELECT * FROM Lehrer` bedeutet: Gib alle Spalten (`*`) aller Datensätze der Tabelle *Lehrer* zurück. Mit `cur.fetchall()` holen wir das Ergebnis ab. Es ist eine Liste, in der jeder Datensatz ein Tupel ist. Mit einer Schleife geben wir die Datensätze nacheinander aus.

In [ ]:
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT *
    FROM Lehrer
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

### Daten einfügen, ändern und löschen

##### INSERT INTO: Daten einfügen

Die Werte stehen in der Reihenfolge der Spalten. Strings stehen in einfachen Hochkommata.

In [ ]:
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("INSERT INTO Lehrer VALUES ('MU', 'Muench')")

con.commit()
con.close()

##### UPDATE: Daten ändern

In [ ]:
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("UPDATE Lehrer SET Lehrer = 'Münch' WHERE Kuerzel = 'MU'")

con.commit()
con.close()

Ohne `WHERE` werden **alle** Datensätze geändert.

##### DELETE FROM: Daten löschen

In [ ]:
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("DELETE FROM Lehrer WHERE Kuerzel = 'MU'")

con.commit()
con.close()

Ohne `WHERE` werden **alle** Datensätze gelöscht.

##### Referentielle Integrität in SQLite

SQLite prüft Fremdschlüssel nur, wenn wir es mit `PRAGMA foreign_keys = ON` einschalten. Dann verweigert es zum Beispiel das Löschen von Dallmeyer, weil der Kurs Bio1 noch auf das Kürzel DA verweist. Das DBMS meldet einen **IntegrityError**, und die Datenbank bleibt unverändert.

**Wichtig:** Bricht eine Zelle zwischen `connect()` und `close()` mit einem Fehler ab, bleibt die Verbindung offen. Unter Windows lässt sich die Datenbankdatei dann nicht löschen oder verschieben, bis wir `con.close()` ausführen oder den Kernel neu starten (**Kernel → Restart Kernel**).

In [ ]:
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("PRAGMA foreign_keys = ON")
cur.execute("DELETE FROM Lehrer WHERE Kuerzel = 'DA'")
con.commit()

con.close()

In [ ]:
# Die Zelle oben bricht mit dem IntegrityError ab, bevor con.close() erreicht wird.
# Diese Zelle schließt die Verbindung, damit die Datei nicht gesperrt bleibt.
con.close()

##### Anmerkung:

PRAGMA ist ein Wort aus dem Griechischen und heißt „Sache“ oder „Handlung“. In der Informatik steht es seit Langem für eine Anweisung an das System selbst. Ein Pragma verarbeitet also keine Daten, sondern stellt ein, wie das System arbeiten soll. 

In SQLite ist PRAGMA ein eigener Befehl, mit dem man interne Einstellungen des DBMS abfragt oder ändert. Er gehört nicht zum SQL-Standard, andere Datenbanksysteme wie MySQL oder PostgreSQL haben dafür andere Befehle.

###  Abfragen auf einer Tabelle

Mit `SELECT` fragen wir Daten ab. Die Grundform ist:

```
SELECT Spalten
FROM Tabelle
WHERE Bedingung
```

- Hinter `SELECT` stehen die Spalten, die wir sehen wollen. `*` steht für **alle** Spalten. Die Auswahl von Spalten heißt **Projektion**.
- Mit `WHERE` wählen wir die Datensätze aus, die eine Bedingung erfüllen. Die Auswahl von Zeilen heißt **Selektion**.


**Beispiel (Projektion)**: Gib die Namen aller Schülerinnen und Schüler aus.

In [ ]:
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT Name, Vorname
    FROM Schueler
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

**Beispiel (Selektion)**: Gib die Daten des Schülers mit der Nummer 121 aus.

In [ ]:
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT *
    FROM Schueler
    WHERE SNr = 121
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

##### Bedingungen in WHERE

| Operator | Bedeutung | Beispiel |
|:--|:--|:--|
| `=`, `<>` | gleich, ungleich | `Kuerzel = 'SC'` |
| `<`, `<=`, `>`, `>=` | Vergleiche | `Stunden >= 4` |
| `AND`, `OR`, `NOT` | Bedingungen verknüpfen | `Kuerzel = 'SC' OR Kuerzel = 'AD'` |
| `LIKE` | Textmuster, `%` steht für beliebig viele Zeichen | `Name LIKE 'R%'` (beginnt mit R) |

Anders als in Python steht in SQL für „gleich“ ein **einfaches** `=`. Strings stehen in einfachen Hochkommata. Bei diesen Werten kommt es, anders als bei den Schlüsselwörtern, auf die **Groß- und Kleinschreibung** an: `WHERE Lehrer = 'schulte'` findet nichts, weil `'Schulte'` gespeichert ist. Nur `LIKE` ignoriert in SQLite die Groß- und Kleinschreibung, allerdings nicht bei Umlauten.

##### ORDER BY, DISTINCT und LIMIT

- `ORDER BY Spalte` **sortiert** das Ergebnis aufsteigend, `ORDER BY Spalte DESC` absteigend.
- `SELECT DISTINCT` entfernt **doppelte Zeilen** aus dem Ergebnis.
- `LIMIT n` begrenzt das Ergebnis auf die ersten n Datensätze. `LIMIT` steht immer am Ende, also nach `ORDER BY`.

**Beispiel:** Schulte unterrichtet zwei LKs, deshalb erscheint das Kürzel SC ohne DISTINCT doppelt.

In [ ]:
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT Kuerzel
    FROM Kurse
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

In [ ]:
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT DISTINCT Kuerzel
    FROM Kurse
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

**Beispiel:** Gib die ersten beiden Schülerinnen und Schüler in alphabetischer Reihenfolge aus.

In [ ]:
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT Name, Vorname
    FROM Schueler
    ORDER BY Name DESC
    LIMIT 2
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

### Abfragen über mehrere Tabellen

Die Informationen sind nach der Normalisierung auf mehrere Tabellen verteilt. Um sie wieder zusammenzuführen, nennen wir hinter `FROM` **mehrere Tabellen** und verknüpfen sie in `WHERE` über **Fremdschlüssel und Primärschlüssel**. Das nennt man einen **Verbund** (Join).

Kommt ein Spaltenname in mehreren Tabellen vor, schreiben wir den **Tabellennamen** davor, z. B. `Kurse.Kuerzel`.



`FROM Kurse, Lehrer` bildet zunächst **alle Kombinationen** aus einem Kurs und einer Lehrkraft, bei 5 Kursen und 4 Lehrkräften also 20 Zeilen.

In [ ]:
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT *
    FROM Kurse, Lehrer
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

 Die Bedingung `Kurse.Kuerzel = Lehrer.Kuerzel` behält davon nur die Zeilen, die wirklich zusammengehören.

In [ ]:
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT *
    FROM Kurse, Lehrer
    WHERE Kurse.Kuerzel = Lehrer.Kuerzel
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

**Beispiel:** Gesucht sind alle Kurse von Schulte oder Adam, diesmal über den Namen der Lehrkraft.

In [ ]:
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT KID, Lehrer
    FROM Kurse, Lehrer
    WHERE Kurse.Kuerzel = Lehrer.Kuerzel 
        AND (Lehrer = 'Adam' OR Lehrer = 'Schulte')
   
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

Die Klammern um die OR-Bedingung sind wichtig, denn `AND` wird vor `OR` ausgewertet.

**Beispiel**: Gib für jeden Schüler (sortiert nach Nachnamen), die KursID und die Lehrkraft aus. 

In [ ]:
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT Name, Vorname, LKWahl.KID, Lehrer
    FROM Schueler, LKWahl, Kurse, Lehrer
    WHERE Schueler.SNr = LKWahl.SNr
      AND LKWahl.KID = Kurse.KID
      AND Kurse.Kuerzel = Lehrer.Kuerzel
    ORDER BY Name
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

Für eine übersichtlichere Ausgabe **entpacken** wir jedes Tupel in einzelne Variablen und benutzen einen f-String. `{name:10}` reserviert 10 Zeichen für den Namen, so entstehen gerade Spalten.

In [ ]:
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT Name, Vorname, LKWahl.KID, Lehrer
    FROM Schueler, LKWahl, Kurse, Lehrer
    WHERE Schueler.SNr = LKWahl.SNr
      AND LKWahl.KID = Kurse.KID
      AND Kurse.Kuerzel = Lehrer.Kuerzel
    ORDER BY Name
""")

for name, vorname, kid, lehrer in cur.fetchall():          # unpacking
    print(f'{vorname:10} {name:10} {kid:4} {lehrer:10}')

con.close()

### Verbund mit JOIN

Bisher haben wir bei einem Verbund die Tabellen hinter `FROM` aufgezählt und die **Verknüpfungsbedingung** in `WHERE` geschrieben. Dieselben Abfragen lassen sich auch mit `JOIN … ON` formulieren:

```
SELECT Spalten
FROM Tabelle1
JOIN Tabelle2 ON Verknüpfungsbedingung
WHERE weitere Bedingungen
```

- Hinter `JOIN` steht die Tabelle, die dazukommt.
- Hinter `ON` steht, **wie** sie mit den bisherigen Tabellen verknüpft ist, also Fremdschlüssel = Primärschlüssel.
- In `WHERE` stehen dann nur noch die Bedingungen, mit denen wir **Datensätze auswählen**.

Das Ergebnis ist dasselbe wie mit `WHERE`. Der Vorteil: Verknüpfung und Auswahl sind getrennt. Bei jeder Tabelle sieht man sofort, über welche Spalten sie angebunden ist.

`JOIN` heißt ausführlich auch `INNER JOIN`. Im Ergebnis erscheinen nur Datensätze, die in beiden Tabellen einen passenden Partner haben.

Beispiel: Gesucht sind alle Kurse von Schulte oder Adam


In [ ]:
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT KID, Lehrer
    FROM Kurse
    JOIN Lehrer ON Kurse.Kuerzel = Lehrer.Kuerzel
    WHERE Lehrer = 'Schulte' OR Lehrer = 'Adam'
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

Die Reihenfolge der Schlüssel bei ON ist egal. In den Beispielen wählen wir zuerst das Feld der bereits vorhandenen Tabelle, dann das Feld der neu hinzukommenden Tabelle.


**Beispiel:** Gib für jeden Schüler (sortiert nach Nachnamen), die KursID und die Lehrkraft aus.  


In [ ]:
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT Name, Vorname, LKWahl.KID, Lehrer
    FROM Schueler
    JOIN LKWahl ON Schueler.SNr = LKWahl.SNr
    JOIN Kurse ON LKWahl.KID = Kurse.KID
    JOIN Lehrer ON Kurse.Kuerzel = Lehrer.Kuerzel
    ORDER BY Name
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

#### Aliase

Wir können für Tabellen Aliase vergeben und uns dann mit den Aliasen auf diese Tabellen beziehen. Im Beispiel werden kleine Buchstaben a, b, c, d
als Aliase für die Tabellen vergeben.



In [ ]:
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT Name, Vorname, b.KID, Lehrer
    FROM Schueler a
    JOIN LKWahl b ON a.SNr = b.SNr
    JOIN Kurse c ON b.KID = c.KID
    JOIN Lehrer d ON c.Kuerzel = d.Kuerzel
    ORDER BY Name
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

### Zählen und Summieren

**Aggregatfunktionen** fassen viele Werte zu **einem einzigen** Wert zusammen.

| Funktion | Wirkung |
|:--|:--|
| `SUM(Spalte)` | addiert alle Werte einer Spalte |
| `COUNT(*)` | zählt die Datensätze im Ergebnis |
| `COUNT(DISTINCT Spalte)` | zählt die verschiedenen Werte einer Spalte |

Mit `WHERE` wählen wir vorher aus, welche Zeilen addiert oder gezählt werden.

Beispiel: Wie viele Wochenstunden LK-Unterricht gibt es insgesamt, und wie viele hat Schulte?

In [ ]:
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT SUM(Stunden)
    FROM Kurse
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

In [ ]:
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT SUM(Stunden)
    FROM Kurse
    WHERE Kuerzel = 'SC'
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

**Beispiel:** Wie viele Schülerinnen und Schüler sind gespeichert, und wie viele haben den Kurs Ph1 gewählt?

In [ ]:
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT COUNT(*)
    FROM Schueler
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

In [ ]:
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT COUNT(*)
    FROM LKWahl
    WHERE KID = 'Ph1'
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

**Beispiel:** Wie viele verschiedene Lehrkräfte unterrichten einen LK? Mit `COUNT(DISTINCT …)` wird SC nur einmal gezählt.

In [ ]:
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT COUNT(DISTINCT Kuerzel)
    FROM Kurse
""")

for datensatz in cur.fetchall():
    print(datensatz)

con.close()

Das Ergebnis ist ein Tupel mit einem einzigen Wert, z. B. `(4,)`. Mit `cur.fetchall()[0][0]` erhalten wir die Zahl selbst und können sie in einen Satz einbauen:

In [ ]:
con = sqlite3.connect("lkwahl.db")
cur = con.cursor()

cur.execute("""
    SELECT COUNT(*)
    FROM Schueler
""")

anzahl = cur.fetchall()[0][0]
print(f'Es sind {anzahl} Schülerinnen und Schüler gespeichert.')

con.close()